<a href="https://colab.research.google.com/github/tim-parisi/data-science-project/blob/main/Data_Science_Project_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Science Project

Team Members:
Ignacio, Timothy, Jamie, Megan

...




# Data Exploration and Cleaning

In [29]:
# Libraries
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import unittest

## Dataset B (Student Engagement)

In [30]:
# Download dataset
df_student = pd.read_csv('/content/UK online student engagement.csv')
df_student

,Unnamed: 0,External,Year,session 1,session 2,test 1,session 3,session 4,test 2,session 5,test 3,session 6,ind cw,group cw,final grade,fourm Q,fourm A,office hour visits,droupout
0,0,N,third,13,20,F,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,1,Y
1,1,N,third,71,77,A,40.0,67.0,A,69.0,A,41.0,A,A,D,13,4,4,N
2,2,Y,first,32,62,F,76.0,77.0,A,35.0,F,93.0,F,F,B,16,9,3,N
3,3,Y,first,8,4,F,46.0,36.0,C,93.0,B,79.0,A,D,B,22,4,6,N
4,4,Y,first,6,1,F,23.0,33.0,D,90.0,C,82.0,C,A,C,19,9,7,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11244,11244,Y,second,7,30,D,4.0,35.0,F,28.0,B,91.0,D,D,F,19,7,4,N
11245,11245,Y,third,59,63,A,90.0,40.0,A,57.0,F,93.0,D,C,A,13,6,5,N
11246,11246,Y,third,25,59,C,82.0,33.0,F,77.0,B,56.0,B,F,A,12,9,5,N
11247,11247,N,third,12,13,D,40.0,43.0,D,57.0,B,23.0,D,C,D,12,6,6,N


### Standardizing Data Format

#### Standardize column names

In [31]:
'''Standardizing Column Names'''
# Removing Unnamed: 0 column
df_student = df_student.drop(columns=['Unnamed: 0'])
# Lowercase and remove whitespace from column names
df_student.columns = [col.lower().replace(' ', '_') for col in df_student.columns]
df_student = df_student.rename(columns={'fourm_q':'forum_q', 'fourm_a':'forum_a', 'droupout':'dropout'})

#### Standardize Column Datatypes

In [32]:
non_cont_cols = [name for name in df_student.columns if ('session' not in name)]
for col in non_cont_cols:
    df_student[col] = df_student[col].astype(object)
df_student.dtypes

,0
external,object
year,object
session_1,int64
session_2,int64
test_1,object
session_3,float64
session_4,float64
test_2,object
session_5,float64
test_3,object


#### Check for mismatches

In [33]:
'''Check for mismatched data values'''
for col in df_student.columns:
    if df_student[col].dtype == 'object':
        col_values = df_student[col].unique()
        print(col_values)
  # No values are 'mismatched' (first/second/third, Y/N, and A/B/C/D/F are all present/consistent/no whitespace
  # NaN values will be handled in different phase)

['N' 'Y']
['third' 'first' 'second']
['F' 'A' 'C' 'D' 'B']
[nan 'A' 'C' 'D' 'F' 'B']
[nan 'A' 'F' 'B' 'C' 'D']
[nan 'A' 'F' 'C' 'B' 'D']
[nan 'A' 'F' 'D' 'B' 'C']
[nan 'D' 'B' 'C' 'A' 'F']
[1 13 16 22 19 6 12 4 11 0 8 15 7 18 23 14 3 2 9 10 17 29 24 5 25 20 28 26
 21 27]
[0 4 9 3 8 6 2 1 12 7 22 11 5 17 24 14 15 20 19 10 18 16 13 21 23]
[1 4 3 6 7 0 2 5 8 9]
['Y' 'N']


#### Moving values to more standardized datatypes

In [34]:
# Converting Y/N to True/False, first/second/third to 1/2/3, A/B/C/D/F to 5/4/3/2/1
df_student = df_student.replace({'Y': True, 'N': False, 'first': 1, 'second': 2, 'third': 3, 'A': 5, 'B': 4, 'C': 3, 'D': 2, 'F': 1})

<ipython-input-34-7b7794864b18>:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_student = df_student.replace({'Y': True, 'N': False, 'first': 1, 'second': 2, 'third': 3, 'A': 5, 'B': 4, 'C': 3, 'D': 2, 'F': 1})


#### Check for poorly formatted numbers

In [35]:
unformatted_number_count = 0
for val in col_values:
    try:
        test_val = float(val)
        # Checking for decimals
        if val != round(val):
            print(f'Decimal Number: {val}')
            unformatted_number_count += 1
        # Checking for impossible percentages
        if val > 100 or val < 0:
            print(f'Incorrect Number: {val}')
            unformatted_number_count += 1
    except ValueError: #for NaN numbers
        pass
print(f'Unformatted Numbers: {unformatted_number_count}')
# No decimal numbers, no apparent issues with mismatches

Unformatted Numbers: 0


#### Check for duplicates

In [36]:
'''Checking for duplicates'''
duplicates = df_student[df_student.duplicated(keep=False)]
print(duplicates)
# Most likely different students, it would be more suspicious of duplicates were they not all early dropouts or if they were sequential

       external  year  session_1  session_2  test_1  session_3  session_4  \
1127       True     2         14          2       1        NaN        NaN   
1440       True     2         14          2       1        NaN        NaN   
3116       True     1         17         19       1        NaN        NaN   
5592       True     1         17         19       1        NaN        NaN   
7713       True     1         12         14       1        NaN        NaN   
10256      True     1         12         14       1        NaN        NaN   

       test_2  session_5  test_3  session_6  ind_cw  group_cw  final_grade  \
1127      NaN        NaN     NaN        NaN     NaN       NaN          NaN   
1440      NaN        NaN     NaN        NaN     NaN       NaN          NaN   
3116      NaN        NaN     NaN        NaN     NaN       NaN          NaN   
5592      NaN        NaN     NaN        NaN     NaN       NaN          NaN   
7713      NaN        NaN     NaN        NaN     NaN       NaN         

### Adding Session Dropped Column

In [37]:
def get_drop(row):
    if row['dropout'] == False:
        return 5
    if pd.isna(row['session_3']):
        return 1
    if pd.isna(row['session_5']):
        return 2
    if pd.isna(row['session_6']):
        return 3
    else:
        return 4
df_student['session_dropped'] = df_student.apply(get_drop, axis=1)
df_student

,external,year,session_1,session_2,test_1,session_3,session_4,test_2,session_5,test_3,session_6,ind_cw,group_cw,final_grade,forum_q,forum_a,office_hour_visits,dropout,session_dropped
0,False,3,13,20,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0,1,True,1
1,False,3,71,77,5,40.0,67.0,5.0,69.0,5.0,41.0,5.0,5.0,2.0,13,4,4,False,5
2,True,1,32,62,1,76.0,77.0,5.0,35.0,1.0,93.0,1.0,1.0,4.0,16,9,3,False,5
3,True,1,8,4,1,46.0,36.0,3.0,93.0,4.0,79.0,5.0,2.0,4.0,22,4,6,False,5
4,True,1,6,1,1,23.0,33.0,2.0,90.0,3.0,82.0,3.0,5.0,3.0,19,9,7,False,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11244,True,2,7,30,2,4.0,35.0,1.0,28.0,4.0,91.0,2.0,2.0,1.0,19,7,4,False,5
11245,True,3,59,63,5,90.0,40.0,5.0,57.0,1.0,93.0,2.0,3.0,5.0,13,6,5,False,5
11246,True,3,25,59,3,82.0,33.0,1.0,77.0,4.0,56.0,4.0,1.0,5.0,12,9,5,False,5
11247,False,3,12,13,2,40.0,43.0,2.0,57.0,4.0,23.0,2.0,3.0,2.0,12,6,6,False,5


In [38]:
df_check = df_student[(pd.isna(df_student['final_grade'])) & ~(pd.isna(df_student['session_6']))]
df_check.size

0

In [39]:
session_cols = [name for name in df_student.columns if ('session' in name)]
for col in session_cols:
    print(f"Column '{col}':")
    print(df_student[col].describe())

Column 'session_1':
count    11249.000000
mean        38.489732
std         30.059852
min          0.000000
25%         10.000000
50%         32.000000
75%         64.000000
max         99.000000
Name: session_1, dtype: float64
Column 'session_2':
count    11249.000000
mean        40.536848
std         28.072742
min          0.000000
25%         14.000000
50%         39.000000
75%         63.000000
max         99.000000
Name: session_2, dtype: float64
Column 'session_3':
count    10309.000000
mean        46.549811
std         27.782394
min          0.000000
25%         23.000000
50%         48.000000
75%         69.000000
max         99.000000
Name: session_3, dtype: float64
Column 'session_4':
count    10309.000000
mean        43.053545
std         23.244112
min          0.000000
25%         26.000000
50%         42.000000
75%         57.000000
max         99.000000
Name: session_4, dtype: float64
Column 'session_5':
count    9063.000000
mean       49.063224
std        25.111657
min  

#### Checking for high correlation columns

In [40]:
'''High Correlation Columns'''
cor_target = abs(df_student.corr(numeric_only=True))
cor_target[cor_target>0.4]

,external,year,session_1,session_2,test_1,session_3,session_4,test_2,session_5,test_3,session_6,ind_cw,group_cw,final_grade,forum_q,forum_a,office_hour_visits,dropout,session_dropped
external,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
session_1,NaN,NaN,1.000000,0.536096,NaN,0.446564,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
session_2,NaN,NaN,0.536096,1.000000,NaN,0.451841,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
test_1,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
session_3,NaN,NaN,0.446564,0.451841,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
session_4,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
test_2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
session_5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
test_3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Combining related columns

In [41]:
forum_q_avg = df_student['forum_q'].mean()
forum_a_avg = df_student['forum_a'].mean()
office_hour_visits_avg = df_student['office_hour_visits'].mean()
df_student = df_student.assign(outside_interaction=df_student['forum_q']/forum_q_avg + df_student['forum_a']/forum_a_avg + df_student['office_hour_visits']/office_hour_visits_avg)
df_student = df_student.assign(first_sessions=df_student['session_1']+df_student['session_2'])
df_student = df_student.drop(columns=['forum_q', 'forum_a', 'office_hour_visits','session_1','session_2'])
df_student

,external,year,test_1,session_3,session_4,test_2,session_5,test_3,session_6,ind_cw,group_cw,final_grade,dropout,session_dropped,outside_interaction,first_sessions
0,False,3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,1,0.493747,33
1,False,3,5,40.0,67.0,5.0,69.0,5.0,41.0,5.0,5.0,2.0,False,5,3.847433,148
2,True,1,1,76.0,77.0,5.0,35.0,1.0,93.0,1.0,1.0,4.0,False,5,4.754416,94
3,True,1,1,46.0,36.0,3.0,93.0,4.0,79.0,5.0,2.0,4.0,False,5,5.742345,12
4,True,1,1,23.0,33.0,2.0,90.0,3.0,82.0,3.0,5.0,3.0,False,5,6.599773,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11244,True,2,2,4.0,35.0,1.0,28.0,4.0,91.0,2.0,2.0,1.0,False,5,5.154543,37
11245,True,3,5,90.0,40.0,5.0,57.0,1.0,93.0,2.0,3.0,5.0,False,5,4.564431,122
11246,True,3,3,82.0,33.0,1.0,77.0,4.0,56.0,4.0,1.0,5.0,False,5,4.964123,84
11247,False,3,2,40.0,43.0,2.0,57.0,4.0,23.0,2.0,3.0,2.0,False,5,4.798916,25


## Splitting Dataset

### Dividing Interaction Columns

In [42]:
def divide_interaction_first(row):
    if row['session_dropped'] < 5:
        return row['outside_interaction'] / row['session_dropped']
    else:
        return row['outside_interaction'] / 4
def divide_interaction_second(row):
    if row['session_dropped'] < 2:
        raise ValueError('Predicting for a timeframe after the student dropped')
    return divide_interaction_first(row) * 2
def divide_interaction_third(row):
    if row['session_dropped'] < 3:
        raise ValueError('Predicting for a timeframe after the student dropped')
    return divide_interaction_first(row) * 3

In [57]:
'''Testing'''
class TestDivideInteraction(unittest.TestCase):
    def test_first_session_dropped(self):
        test_student_full = {'outside_interaction': 1, 'session_dropped': 1}
        self.assertEqual(divide_interaction_first(test_student_full), test_student_full['outside_interaction'])

    def test_second_session_dropped(self):
        test_student_full = {'outside_interaction': 1, 'session_dropped': 2}
        self.assertEqual(divide_interaction_first(test_student_full), test_student_full['outside_interaction']/2)
        self.assertEqual(divide_interaction_second(test_student_full), test_student_full['outside_interaction'])

    def test_third_session_dropped(self):
        test_student_full = {'outside_interaction': 0.6, 'session_dropped': 3}
        self.assertEqual(divide_interaction_first(test_student_full), test_student_full['outside_interaction']/3)
        self.assertEqual(divide_interaction_second(test_student_full), test_student_full['outside_interaction']*2/3)
        self.assertEqual(divide_interaction_third(test_student_full), test_student_full['outside_interaction'])

    def test_fourth_session_dropped(self):
        test_student_full = {'outside_interaction': 1, 'session_dropped': 4}
        self.assertEqual(divide_interaction_first(test_student_full), test_student_full['outside_interaction']/4)
        self.assertEqual(divide_interaction_second(test_student_full), test_student_full['outside_interaction']*2/4)
        self.assertEqual(divide_interaction_third(test_student_full), test_student_full['outside_interaction']*3/4)

    def test_fifth_session_dropped(self):
        test_student_full = {'outside_interaction': 1, 'session_dropped': 5}
        self.assertEqual(divide_interaction_first(test_student_full), test_student_full['outside_interaction']/4)
        self.assertEqual(divide_interaction_second(test_student_full), test_student_full['outside_interaction']*2/4)
        self.assertEqual(divide_interaction_third(test_student_full), test_student_full['outside_interaction']*3/4)

    def test_zeros(self):
        test_student_zeros = {'outside_interaction': 0, 'session_dropped': 5}
        self.assertEqual(divide_interaction_first(test_student_zeros),0)
        self.assertEqual(divide_interaction_second(test_student_zeros),0)
        self.assertEqual(divide_interaction_third(test_student_zeros),0)

    def test_out_of_bounds(self):
        test_student_zeros = {'outside_interaction': 0, 'session_dropped': 1}
        with self.assertRaises(Exception) as context:
            divide_interaction_second(test_student_zeros)
        self.assertTrue(type(context.exception) == ValueError)
        with self.assertRaises(Exception) as context:
            divide_interaction_third(test_student_zeros)
        self.assertTrue(type(context.exception) == ValueError)

unittest.main(argv=[''], verbosity=2, exit=False)

test_fifth_session_dropped (__main__.TestDivideInteraction.test_fifth_session_dropped) ... ok
test_first_session_dropped (__main__.TestDivideInteraction.test_first_session_dropped) ... ok
test_fourth_session_dropped (__main__.TestDivideInteraction.test_fourth_session_dropped) ... ok
test_out_of_bounds (__main__.TestDivideInteraction.test_out_of_bounds) ... ok
test_second_session_dropped (__main__.TestDivideInteraction.test_second_session_dropped) ... ok
test_third_session_dropped (__main__.TestDivideInteraction.test_third_session_dropped) ... ok
test_zeros (__main__.TestDivideInteraction.test_zeros) ... ok

----------------------------------------------------------------------
Ran 7 tests in 0.015s

OK


### First Dropout

In [ ]:
first_student = df_student
first_student = first_student.assign(dropout=first_student['session_dropped'] < 2)
first_student['outside_interaction'] = first_student.apply(divide_interaction_first, axis=1)
first_student = first_student.drop(columns=['session_dropped', 'session_3', 'session_4', 'test_2', 'session_5', 'test_3', 'session_6', 'ind_cw', 'group_cw', 'final_grade'])
first_student

#### Dummy Set

In [ ]:
first_dummy = first_student.replace({True: 1, False: 0})
first_dummy

#### PCA

In [ ]:
scaler = StandardScaler()
pca = PCA(n_components=5)
X_scaled = scaler.fit_transform(first_dummy.drop(columns=['dropout']))
X_pca = pca.fit_transform(X_scaled)
plt.plot(pca.explained_variance_ratio_, marker='o')
plt.show()
total_components = 4
pca_1 = PCA(n_components=total_components)
X_pca_1 = pca_1.fit_transform(X_scaled)
print(f'Total variance with {total_components} components: {pca_1.explained_variance_ratio_.sum():.4f}%')

### Second Dropout

In [ ]:
second_student = df_student
second_student = second_student.assign(dropout=second_student['session_dropped'] < 3)
second_student = second_student[second_student['session_dropped'] > 1]
second_student['outside_interaction'] = second_student.apply(divide_interaction_second, axis=1)
second_student = second_student.drop(columns=['session_dropped', 'session_5', 'test_3', 'session_6', 'ind_cw', 'group_cw', 'final_grade'])
second_student

#### Dummy Set

In [ ]:
second_dummy = second_student.replace({True: 1, False: 0})
second_dummy

#### PCA

In [ ]:
scaler = StandardScaler()
pca = PCA(n_components=8)
X_scaled = scaler.fit_transform(second_dummy.drop(columns=['dropout']))
X_pca = pca.fit_transform(X_scaled)
plt.plot(pca.explained_variance_ratio_, marker='o')
plt.show()
total_components = 6
pca_2 = PCA(n_components=total_components)
X_pca_2 = pca_2.fit_transform(X_scaled)
print(f'Total variance with {total_components} components: {pca_2.explained_variance_ratio_.sum():.4f}%')

### Third Dropout

In [ ]:
third_student = df_student
third_student = third_student.assign(dropout=third_student['session_dropped'] < 4)
third_student = third_student[third_student['session_dropped'] > 2]
third_student['outside_interaction'] = third_student.apply(divide_interaction_third, axis=1)
third_student = third_student.drop(columns=['session_dropped', 'session_6', 'ind_cw', 'group_cw', 'final_grade'])
third_student

#### Dummy Set

In [ ]:
third_dummy = third_student.replace({True: 1, False: 0})
third_dummy

#### PCA

In [ ]:
scaler = StandardScaler()
pca = PCA(n_components=10)
X_scaled = scaler.fit_transform(third_dummy.drop(columns=['dropout']))
X_pca = pca.fit_transform(X_scaled)
plt.plot(pca.explained_variance_ratio_, marker='o')
plt.show()
total_components = 7
pca_3 = PCA(n_components=total_components)
X_pca_3 = pca_3.fit_transform(X_scaled)
print(f'Total variance with {total_components} components: {pca_3.explained_variance_ratio_.sum():.4f}%')

# Unit Tests

# Heatmap/Correlation Matrix




# Modeling

### Visualizing Datasets

In [ ]:
datasets = {1: (X_pca_1, first_dummy['dropout']), 2: (X_pca_2, second_dummy['dropout']), 3: (X_pca_3, third_dummy['dropout'])}
graphing_datasets = X_pca_1[:,0], X_pca_2[:1]
fig, axes = plt.subplots(1, 3)
fig.suptitle('SVM Datasets')
for ix, (X, y) in enumerate(datasets.values(), start=0):
    axes[ix].scatter(X[:,0], X[:,1], c=y)
    axes[ix].set_title(f'Dropout {ix+1}')
plt.show()